# PhishExplain na DGX Spark: rodada completa (Fases 7 e 8)

Este notebook faz tudo o que falta e termina com todas as tabelas de resultados:

1. confere o ambiente e sobe o professor (`Qwen/Qwen3.5-27B`) no vLLM;
2. mede a vazão do professor e estima o tempo total;
3. traduz, gera os sintéticos, filtra e anota (7.1);
4. separa treino e validação, exporta as conversas e checa vazamento (Fase 4);
5. roda o professor, o baseline e os três alunos em zero-shot no teste (7.2);
6. treina os três alunos com LoRA e roda as previsões com os adaptadores (7.3 a 7.5);
7. julga as explicações com um modelo de outra família (8.2);
8. monta as tabelas finais em `reports/resultados.md` (8.1 e 8.3).

**Antes de abrir este notebook (Fase 6), na Spark:**

```bash
git clone <repositório> ~/phishexplain && cd ~/phishexplain
rsync -av <pc>:~/phishexplain/data/interim/ data/interim/   # 6.3: só emails_agrupados.parquet é obrigatório
export HF_TOKEN=hf_...          # aceite antes as licenças do Gemma 4 e do Mistral Small no Hugging Face
./scripts/spark_construir.sh    # 6.1: imagens unsloth-dgx-spark e phishexplain-unsloth:0.1 (demora)
./scripts/spark_container.sh    # 6.2: Jupyter na porta 8888 (túnel: ssh -L 8888:localhost:8888 <spark>)
```

**Como rodar:** `Run All`. Tudo é idempotente: se algo cair, rode de novo; o que já foi feito é pulado.
As saídas do professor de desenvolvimento do PC **não** entram aqui: apague `data/interim/*.jsonl` e
`data/interim/sinteticos_filtrados.parquet` se vieram no `rsync` (a primeira célula faz isso por você, se você deixar).

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

assert os.environ.get("PHISH_PERFIL") == "spark", "suba o container com ./scripts/spark_container.sh"
from src.config import CFG, DADOS_INTERIM, JUIZ_MODELO, RELATORIOS, SAIDAS
from src import servidor

# ---- Configurações da rodada -------------------------------------------------------------------
servidor.VLLM_IMAGEM = "nvcr.io/nvidia/vllm:26.09-py3"  # precisa de vLLM >= 0.17; alternativa: "vllm/vllm-openai:cu130-nightly"
PROFESSOR = CFG.professor_modelo      # plano B, se a estimativa passar de uma noite: "Qwen/Qwen3.5-35B-A3B"
ALUNOS = ["gemma4-e2b", "qwen35-4b", "s5-nano"]  # ordem do treino: o menor primeiro, para achar bugs cedo
APAGAR_SAIDAS_DO_PC = True            # saídas do professor de desenvolvimento nunca entram no treino
# -------------------------------------------------------------------------------------------------

def rodar(*args, **env):
    """Roda `python -m <args>` mostrando a saída; para o notebook se der erro."""
    print("$ python -m", " ".join(args), flush=True)
    r = subprocess.run([sys.executable, "-m", *args], env=os.environ | {k: str(v) for k, v in env.items()})
    assert r.returncode == 0, f"falhou: {args}"

if APAGAR_SAIDAS_DO_PC and not (DADOS_INTERIM / ".rodada_spark").exists():
    for f in [*DADOS_INTERIM.glob("*.jsonl"), DADOS_INTERIM / "sinteticos_filtrados.parquet"]:
        f.unlink(missing_ok=True)
    (DADOS_INTERIM / ".rodada_spark").touch()
print(CFG)

## 6.2 e 6.4: ambiente e professor no vLLM

In [ ]:
rodar("src.ambiente")
import docker
print("Docker:", docker.from_env().version()["Version"], "| imagem do vLLM:", servidor.VLLM_IMAGEM)
docker.from_env().images.pull(servidor.VLLM_IMAGEM)
url = servidor.subir(PROFESSOR, servidor.PROFESSOR, nome=CFG.professor_modelo)
rodar("src.teacher.cliente")

### Vazão do professor

Anota uma amostra de 64 e-mails (que já conta para a rodada) e estima o tempo total.
Se passar de uma noite (~12 h), troque `PROFESSOR` pelo plano B na primeira célula e rode de novo.

In [ ]:
from src.config import SINTETICOS_META
from src.teacher import anotar, traduzir
inicio = time.time()
rodar("src.teacher.traduzir", PHISH_LIMITE=64)
por_email = (time.time() - inicio) / 64
n_traducoes = len(traduzir.selecionar(False)) + len(traduzir.selecionar(True))
n_anotacoes = len(anotar.selecionar(False)) + n_traducoes + SINTETICOS_META  # nativos + traduzidos + sintéticos (+ teste)
total = n_traducoes + SINTETICOS_META + n_anotacoes
print(f"{n_traducoes} traduções, {SINTETICOS_META} sintéticos, {n_anotacoes} anotações")
print(f"{por_email:.2f} s por e-mail com {CFG.professor_paralelo} em paralelo -> estimativa total de {total * por_email / 3600:.1f} h")

## 7.1: tradução, sintéticos e anotação em escala

In [ ]:
rodar("src.teacher.traduzir")              # sementes livres + 1 variante por grupo
rodar("src.teacher.traduzir", "--teste")   # as 299 sementes do recorte traduzido
rodar("src.teacher.gerar")                 # 2.000 sintéticos
rodar("src.teacher.filtrar")               # MinHash contra toda a tabela mestre
rodar("src.teacher.anotar")                # treino: nativos + traduções + sintéticos
rodar("src.teacher.anotar", "--teste")     # referência da avaliação (nunca no treino)
rodar("src.teacher.filtrar")               # de novo, para a revisão já trazer as anotações

**Revisão manual:** leia `data/interim/revisao_sinteticos.md` (50 sintéticos com as anotações).
A rodada segue sem esperar, mas revise antes de publicar os resultados.

In [ ]:
from src.teacher.lote import ler
taxas = {n: (len(ler(DADOS_INTERIM / f"{n}.jsonl")), len(ler(DADOS_INTERIM / f"{n}_reprovad{'as' if n != 'sinteticos' else 'os'}.jsonl")))
         for n in ("traducoes", "sinteticos", "anotacoes", "anotacoes_teste")}
display(pd.DataFrame(taxas, index=["aprovados", "reprovados"]).T.assign(taxa=lambda d: d.aprovados / (d.aprovados + d.reprovados)))

## Fase 4: splits, exportação e professor no teste

In [ ]:
rodar("src.splits")
rodar("src.exportar")
rodar("src.prever", "--nome", "professor", "--modelo", CFG.professor_modelo, "--url", url)
servidor.parar()
rodar("src.vazamento")
display(pd.read_csv(RELATORIOS / "vazamento.csv").head(15))

## 5.2 e 7.2: baseline e alunos em zero-shot

In [ ]:
rodar("src.baseline")
for aluno in ALUNOS:
    repo = CFG.alunos[aluno]
    url = servidor.subir(repo, servidor.ALUNO)
    rodar("src.prever", "--nome", f"{aluno}_zeroshot", "--modelo", repo, "--url", url)
servidor.parar()

## 7.3 a 7.5: treino LoRA (o vLLM fica parado: os dois disputam a mesma memória)

In [ ]:
servidor.parar()
for aluno in ALUNOS:
    if (SAIDAS / aluno / "adapter_config.json").exists():
        print(aluno, "já treinado")
        continue
    rodar("src.train", "--aluno", aluno)

for aluno in ALUNOS:
    h = pd.DataFrame(json.loads((SAIDAS / aluno / "historico.json").read_text()))
    ax = h.dropna(subset=["loss"]).plot(x="step", y="loss", title=aluno, figsize=(6, 3))
    if "eval_loss" in h:
        h.dropna(subset=["eval_loss"]).plot(x="step", y="eval_loss", ax=ax, marker="o")

## Previsões com os adaptadores

In [ ]:
for aluno in ALUNOS:
    repo = CFG.alunos[aluno]
    url = servidor.subir(repo, servidor.ALUNO, lora={f"{aluno}-lora": f"outputs/{aluno}"})
    rodar("src.prever", "--nome", f"{aluno}_lora", "--modelo", f"{aluno}-lora", "--url", url)
servidor.parar()

## 8.2: juiz das explicações

Juiz: `JUIZ_MODELO` (Mistral Small 3.2, família diferente de Qwen e Gemma), sem saber qual modelo escreveu.
Depois, dê suas notas aos 30 casos de `reports/calibracao_juiz.md` para calibrar o juiz.

In [ ]:
previsoes = sorted(p for p in (SAIDAS / "previsoes").glob("*.jsonl") if not p.stem.endswith("_falhas") and p.stem != "baseline_tfidf")
url = servidor.subir(JUIZ_MODELO, ["--max-model-len", "16384", "--gpu-memory-utilization", "0.8", "--max-num-seqs", "32",
                                   "--tokenizer-mode", "mistral", "--config-format", "mistral", "--load-format", "mistral"])
rodar("src.juiz", "--url", url, *map(str, previsoes))
servidor.parar()

## 8.1 e 8.3: tabelas de resultados

In [ ]:
from src.relatorio import main as relatorio
pd.set_option("display.max_columns", 30, "display.width", 250)
for titulo, tabela in relatorio().items():
    display(Markdown(f"### {titulo}"))
    display(tabela)

## Depois da rodada

- Fora do container: `sudo chown -R $USER:$USER ~/phishexplain` (o container cria os arquivos como root).
- Commit dos splits (`splits/treino.csv`, `splits/validacao.csv`) e dos relatórios em `reports/`.
- Preencha a tabela de resultados e as limitações do `README.md` a partir de `reports/resultados.md`.
- Fase 9 (opcional): exportar o melhor adaptador para GGUF com `model.save_pretrained_gguf` da Unsloth.